# Marker Extraction: Five Models (Train + Infer)

One RoBERTa-LoRA model per marker type (Action, Actor, Effect, Evidence, Victim).
- **Train:** `Markers-Extraction/train_rehydrated.jsonl` → `Markers-Extraction/models/roberta-{Type}/final`
- **Infer:** Run on `test_rehydrated.jsonl` (or other JSONL), merge predictions from all 5 models → output JSONL with `_id` and `markers`.

Paths are relative to project root (run from `EDA-Rehydrated/notebooks/`).

In [ ]:
import json
import warnings
import torch
import numpy as np
from pathlib import Path
from torch import nn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    RobertaTokenizerFast, RobertaForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
BASE = PROJECT_ROOT
NOTEBOOK_DIR = Path('.').resolve()
TRAIN_FILE = BASE / 'data' / 'train_rehydrated.jsonl'
MARKER_TYPES = ['Action', 'Actor', 'Effect', 'Evidence', 'Victim']
MODEL_NAME = 'roberta-large'
MAX_LENGTH = 512
BATCH_SIZE = 32
LR = 6e-5
EPOCHS = 10
LORA_R = 16

print(f'BASE (Markers-Extraction): {BASE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## Weighted trainer and label/tokenize helpers

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        device = labels.device
        weights = torch.tensor([1.0, 10.0, 10.0], dtype=torch.float, device=device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, 3), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def create_labels(marker_type):
    label_list = ['O', f'B-{marker_type}', f'I-{marker_type}']
    l2id = {lbl: i for i, lbl in enumerate(label_list)}
    id2l = {i: lbl for i, lbl in enumerate(label_list)}
    return l2id, id2l

def tokenize_and_align(examples, tokenizer, marker_type, label_to_id):
    tokenized = tokenizer(examples['text'], truncation=True, padding='max_length',
                          max_length=MAX_LENGTH, return_offsets_mapping=True)
    labels = []
    for i, offsets in enumerate(tokenized['offset_mapping']):
        doc_labels = [label_to_id['O']] * len(offsets)
        for m in examples['markers'][i]:
            if m['type'] == marker_type:
                start, end = m['startIndex'], m['endIndex']
                first = True
                for idx, (t_start, t_end) in enumerate(offsets):
                    if t_start == t_end == 0 or t_start is None:
                        continue
                    if t_start < end and t_end > start:
                        doc_labels[idx] = label_to_id[f'B-{marker_type}'] if first else label_to_id[f'I-{marker_type}']
                        first = False
        labels.append(doc_labels)
    tokenized['labels'] = labels
    return tokenized

## Load data and tokenizer

In [ ]:
print('Loading data...')
with open(TRAIN_FILE) as f:
    data = [json.loads(line) for line in f]
train_idx, val_idx = train_test_split(range(len(data)), test_size=0.1, random_state=42)
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME, add_prefix_space=True)
print(f'Loaded {len(data)} examples')

## Train one model per marker type

In [ ]:
for m_type in MARKER_TYPES:
    print(f"\n>>> TRAINING: {m_type}")
    l2id, id2l = create_labels(m_type)
    train_ds = Dataset.from_list([data[i] for i in train_idx]).map(
        lambda x: tokenize_and_align(x, tokenizer, m_type, l2id), batched=True
    )
    val_ds = Dataset.from_list([data[i] for i in val_idx]).map(
        lambda x: tokenize_and_align(x, tokenizer, m_type, l2id), batched=True
    )
    model = RobertaForTokenClassification.from_pretrained(MODEL_NAME, num_labels=3)
    peft_config = LoraConfig(
        task_type=TaskType.TOKEN_CLS, inference_mode=False, r=LORA_R,
        lora_alpha=32, lora_dropout=0.1,
        target_modules=['query', 'value', 'key', 'dense']
    )
    model = get_peft_model(model, peft_config)
    output_dir = BASE / 'models' / f'roberta-{m_type}'
    output_dir.mkdir(parents=True, exist_ok=True)
    args = TrainingArguments(
        output_dir=str(output_dir), eval_strategy='epoch', save_strategy='epoch',
        learning_rate=LR, per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=1, num_train_epochs=EPOCHS,
        load_best_model_at_end=True, metric_for_best_model='f1', weight_decay=0.01,
        logging_steps=10, report_to='none', fp16=torch.cuda.is_available(),
        ddp_find_unused_parameters=False
    )
    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=2).flatten()
        labs = p.label_ids.flatten()
        mask = labs != -100
        return {'f1': f1_score(labs[mask], preds[mask], average='macro')}
    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(3)]
    )
    trainer.train()
    final_path = output_dir / 'final'
    final_path.mkdir(exist_ok=True)
    trainer.model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    with open(final_path / 'config_labels.json', 'w') as f:
        json.dump({'l2id': l2id, 'id2l': id2l}, f)
    print(f'✓ {m_type} saved to {final_path}')
    del model
    del trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Inference: run all 5 models and merge markers

In [ ]:
TEST_FILE = BASE / 'data' / 'test_rehydrated.jsonl'
if not TEST_FILE.exists():
    TEST_FILE = PROJECT_ROOT / 'data' / 'test_rehydrated.jsonl'
OUTPUT_PATH = NOTEBOOK_DIR / 'submission_5_models.jsonl'

with open(TEST_FILE) as f:
    test_data = [json.loads(line) for line in f]

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME, add_prefix_space=True)
all_results = [[] for _ in range(len(test_data))]

for m_type in MARKER_TYPES:
    print(f'Extracting {m_type}...')
    model_path = BASE / 'models' / f'roberta-{m_type}' / 'final'
    with open(model_path / 'config_labels.json') as f:
        id2l = {int(k): v for k, v in json.load(f)['id2l'].items()}
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = RobertaForTokenClassification.from_pretrained(
        model_path, num_labels=3, ignore_mismatched_sizes=True
    ).to(device)
    model.eval()
    for i, item in enumerate(test_data):
        inputs = tokenizer(
            item['text'], return_tensors='pt', truncation=True,
            max_length=MAX_LENGTH, return_offsets_mapping=True
        ).to(device)
        offsets = inputs.pop('offset_mapping')[0].cpu().numpy()
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
            preds = np.argmax(probs, axis=-1)
        curr_start = None
        for idx, label_id in enumerate(preds):
            label = id2l[label_id]
            o_start, o_end = offsets[idx]
            if o_start == o_end == 0:
                continue
            if label.startswith('B-'):
                if curr_start is not None:
                    all_results[i].append({'startIndex': int(curr_start), 'endIndex': int(prev_end), 'type': m_type})
                curr_start = o_start
                prev_end = o_end
            elif label.startswith('I-') and curr_start is not None:
                prev_end = o_end
            else:
                if curr_start is not None:
                    all_results[i].append({'startIndex': int(curr_start), 'endIndex': int(prev_end), 'type': m_type})
                curr_start = None
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

with open(OUTPUT_PATH, 'w') as f:
    for i, original_item in enumerate(test_data):
        markers = sorted(all_results[i], key=lambda x: x['startIndex'])
        output_item = {'_id': original_item['_id'], 'markers': markers}
        f.write(json.dumps(output_item) + '\n')
print(f'Saved to {OUTPUT_PATH}')